# Idle lifecycle for an httpbin Deployment

This notebook creates an independent httpbin Deployment, observes idle `STOP`, and then switches to `PAUSE` with a full configuration update. `STOP` releases the Sandbox Instance; `PAUSE` preserves instance state for resumption. Because httpbin is stateless, this example observes request resumption and the Deployment summary rather than claiming to test state persistence.

> Copy IDs and the token manually. Time each idle period yourself; the notebook contains no polling scripts. Replace `AGR_ROLE_ARN` with a CAM role ARN that lets AGR pull the target CCR image.

In [ ]:
%env AGR_REGION=ap-shanghai
%env AGR_DOMAIN=tencentags.com
%env AGR_ROLE_ARN=qcs::cam::uin/replace-me:roleName/replace-me
%env HTTPBIN_TOOL_NAME=httpbin-lifecycle-your-name
%env HTTPBIN_DEPLOYMENT_NAME=httpbin-lifecycle-your-name
!agr status

## 1. Create an independent Tool

Replace `your-name` in the names first.

In [ ]:
!agr tool create \
  --region "$AGR_REGION" \
  --tool-name "$HTTPBIN_TOOL_NAME" \
  --tool-type custom \
  --persistent \
  --role-arn "$AGR_ROLE_ARN" \
  --network-configuration '{"NetworkMode":"PUBLIC"}' \
  --custom-configuration '{"Image":"ccr.ccs.tencentyun.com/ags.dev/go-httpbin:v2.25.0","ImageRegistryType":"personal","Command":["/bin/go-httpbin"],"Args":["-host","0.0.0.0","-port","8080"],"Env":[{"Name":"EXCLUDE_HEADERS","Value":"X-Access-Token"}],"Ports":[{"Name":"http","Port":8080,"Protocol":"TCP"}],"Resources":{"CPU":"200m","Memory":"500Mi"},"Probe":{"HttpGet":{"Path":"/status/200","Port":8080,"Scheme":"HTTP"},"ReadyTimeoutMs":30000,"ProbeTimeoutMs":1000,"ProbePeriodMs":3000,"SuccessThreshold":1,"FailureThreshold":10}}' \
  --wait

## 2. Stop after idling

Copy `ToolId`. `MinInstanceCount=0` lets an idle instance leave active capacity; after 30 idle seconds, `STOP` releases it.

In [ ]:
%env HTTPBIN_TOOL_ID=sdt-replace-me
!agr deployment create \
  --region "$AGR_REGION" \
  --deployment-name "$HTTPBIN_DEPLOYMENT_NAME" \
  --tool-id "$HTTPBIN_TOOL_ID" \
  --scaling-configuration '{"MinInstanceCount":0,"MaxInstanceCount":1,"MaxInstanceRequestConcurrency":10}' \
  --lifecycle-configuration '{"IdleTimeoutSeconds":30,"IdleAction":"STOP"}'

## 3. Activate an instance and observe `STOP`

Copy `DeploymentId`, acquire a token, and copy `Data.Response.Response.Token`. Send one request to start an instance, then send no request or connection for at least 30 seconds before running the second observation cell. Reclamation and later startup are asynchronous and may take slightly longer than the configured timeout.

In [ ]:
%env HTTPBIN_DEPLOYMENT_ID=dpl-replace-me
!agr api call AcquireDeploymentToken --region "$AGR_REGION" --request '{"DeploymentId":"'$HTTPBIN_DEPLOYMENT_ID'"}' --output json

In [ ]:
%env HTTPBIN_DEPLOYMENT_TOKEN=dpt-replace-me
!curl --fail-with-body --silent --show-error --header "X-Access-Token: $HTTPBIN_DEPLOYMENT_TOKEN" "https://8080-$HTTPBIN_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/get"

Stop sending requests for at least 30 seconds, then run the query and request below. `get` shows the Deployment configuration and capacity summary; the following request starts capacity again when necessary.

In [ ]:
!agr deployment get "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"
!curl --fail-with-body --silent --show-error --header "X-Access-Token: $HTTPBIN_DEPLOYMENT_TOKEN" "https://8080-$HTTPBIN_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/get"

## 4. Switch to pausing after idle

A lifecycle update replaces the object in full, so both the timeout and action are supplied. After updating, send one request so the current instance becomes active under the new policy. Idle again for at least 30 seconds, then observe the resume request. The core `PAUSE` guarantee is preserved instance state, not a fixed resume latency.

In [ ]:
!agr deployment update "$HTTPBIN_DEPLOYMENT_ID" \
  --region "$AGR_REGION" \
  --lifecycle-configuration '{"IdleTimeoutSeconds":30,"IdleAction":"PAUSE"}'
!curl --fail-with-body --silent --show-error --header "X-Access-Token: $HTTPBIN_DEPLOYMENT_TOKEN" "https://8080-$HTTPBIN_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/get"

Stop sending requests again for at least 30 seconds, then run:

In [ ]:
!agr deployment get "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"
!curl --fail-with-body --silent --show-error --header "X-Access-Token: $HTTPBIN_DEPLOYMENT_TOKEN" "https://8080-$HTTPBIN_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/get"

## 5. Clean up

In [ ]:
!agr deployment delete "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr instance list --tool-id "$HTTPBIN_TOOL_ID" --region "$AGR_REGION"

A paused instance may remain visible after the `PAUSE` experiment. For every non-`STOPPED` instance, copy its ID. In a new cell, run `%env HTTPBIN_INSTANCE_ID=replace-me`, followed by `!agr instance delete "$HTTPBIN_INSTANCE_ID" --region "$AGR_REGION" --yes --wait`. Then delete the Tool.

In [ ]:
!agr tool delete "$HTTPBIN_TOOL_ID" --region "$AGR_REGION" --yes --wait